# Fraud detection with CatBoost
**Pipeline:** data processing → EDA → feature engineering → load the trained model → predictions.

- **Model:** 3 CatBoost classifiers (seeds 0, 1, 2), trained in `01_final_catboost.ipynb` on all 20,000 training rows and saved to `model.pkl`. **This notebook does no training.**
- **Inputs:** `model.pkl`, the test file (`DATATHON_INPUT_PATH`, else `test.csv`), and optionally `train.csv`, used only for the EDA (skipped if absent).
- **Output:** `prediction`, one row per test row in the test file's order (`DATATHON_OUTPUT_PATH`, else `predictions.csv`); `id, prediction` when the test file has ids.

## 1. Data processing
### 1.1 Load the data
Reads the test file, and the labelled training file if it is available.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#52514e", "#898781"   # colour-blind-safe pair (validated)
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 10, "axes.titlelocation": "left", "font.size": 9, "legend.frameon": False,
})

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH") or next(
    (p for p in ["test.csv", "../data/Track 2 Testing Dataset.csv"] if os.path.exists(p)), "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")
TRAIN_PATH = next((p for p in ["train.csv", "Track 2 Training Dataset.csv", "../data/Track 2 Training Dataset.csv"] if os.path.exists(p)), None)

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Test file {TEST_PATH!r} not found. Set DATATHON_INPUT_PATH, or put the competition CSVs "
                            "in ../data/ (they are not distributed with the repo; see ../data/README.md).")
test = pd.read_csv(TEST_PATH)
train = pd.read_csv(TRAIN_PATH) if TRAIN_PATH else None
HAS_TRAIN = train is not None                      # the EDA needs the labels; prediction does not
print("Test:", test.shape, "| Train:", train.shape if HAS_TRAIN else "not found, EDA skipped")

Test: (12000, 11) | Train: (20000, 12)


### 1.2 Data-quality audit
The share of blanks per column, duplicates and value ranges. The training data is clean (0.5–0.8% blanks per column, no duplicates); the test set has about 3× more blanks.

In [2]:
audit = pd.DataFrame({"test": test.isna().mean() * 100})
if HAS_TRAIN:
    audit.insert(0, "train", train.isna().mean() * 100)
    print("Duplicate ids:", train.id.duplicated().sum(), "| duplicate rows:", train.drop(columns="id").duplicated().sum())
display(audit.round(2).T.rename_axis("% blank"))
display((train if HAS_TRAIN else test).describe().T[["min", "50%", "max"]])

Duplicate ids: 0 | duplicate rows: 0


,id,transaction_amount,transaction_hour,merchant_category,country,transaction_channel,transactions_last_24h,spend_last_24h,account_age,new_device,transactions_last_1h
% blank,,,,,,,,,,,
train,0.0,0.0,0.0,0.56,0.53,0.62,0.80,0.66,0.68,0.62,0.71
test,0.0,0.0,0.0,2.16,2.23,2.37,2.44,2.22,2.20,2.44,2.04


,min,50%,max
transaction_amount,2.5,75.125,9000.00
transaction_hour,0.0,11.000,23.00
transactions_last_24h,1.0,4.000,27.00
spend_last_24h,0.0,217.515,15670.31
account_age,5.0,790.000,4000.00
new_device,0.0,0.000,1.00
transactions_last_1h,0.0,1.000,9.00
fraud,0.0,0.000,1.00


### 1.3 Cleaning
`sanitise` hardens the pipeline for unseen, messier data. It changes nothing on the current files.
- Columns are selected by name: a missing column becomes blank, and extra columns are ignored.
- Text, ±infinity and impossible values (negative amounts/counts/ages, an hour outside 0–23, `new_device` other than 0/1) become blank.
- The 1-hour count is capped at the 24-hour count.
- Category spelling is normalised to training's (`" sg"` → `"SG"`). Unseen levels become blank.

Blanks are **not** zero-filled: CatBoost handles missing values natively.

In [3]:
CATS = ["merchant_category", "country", "transaction_channel"]
RAW_COLS = ["transaction_amount", "transaction_hour", "merchant_category", "country", "transaction_channel",
            "transactions_last_24h", "spend_last_24h", "account_age", "new_device", "transactions_last_1h"]
NUMERIC_RAW = [c for c in RAW_COLS if c not in CATS]
NON_NEGATIVE = ["transaction_amount", "spend_last_24h", "transactions_last_1h", "transactions_last_24h", "account_age"]
LEVELS = {                                         # category levels seen in training
    "merchant_category": ["retail", "gaming", "entertainment", "utilities", "food_delivery", "electronics",
                          "grocery", "travel", "transport", "cash_transfer", "luxury"],
    "country": ["SG", "GB", "TH", "ID", "PH", "AU", "US", "VN", "MY", "JP"],
    "transaction_channel": ["ecommerce", "mobile_app", "card_present", "bank_transfer"],
}
CAT_LEVELS = {c: {v.lower(): v for v in LEVELS[c]} for c in CATS}   # trimmed lower-case spelling -> training spelling

def sanitise(d):
    d = d.reindex(columns=RAW_COLS).copy()                                 # select columns by name; missing ones become NaN
    for c in NUMERIC_RAW:
        d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in NON_NEGATIVE:
        d[c] = d[c].where(d[c] >= 0)                                       # negative -> NaN
    d["transaction_hour"] = d.transaction_hour.where(d.transaction_hour.between(0, 23))
    d["new_device"] = d.new_device.where(d.new_device.isin([0, 1]))
    over = d.transactions_last_1h > d.transactions_last_24h                # 1h count can't exceed the 24h count
    d.loc[over, "transactions_last_1h"] = d.loc[over, "transactions_last_24h"]
    for c in CATS:                                                         # normalise spelling; unseen levels -> NaN -> "Unknown"
        d[c] = d[c].astype("string").str.strip().str.lower().map(CAT_LEVELS[c]).astype(object)
    return d

test_clean = sanitise(test)
train_clean = sanitise(train).assign(fraud=train.fraud) if HAS_TRAIN else None
print("Test blanks before / after cleaning:", test.reindex(columns=RAW_COLS).isna().sum().sum(), "/", test_clean.isna().sum().sum())

Test blanks before / after cleaning: 2173 / 2173


## 2. Exploratory data analysis: where does fraud hide?
The story in six steps: fraud is rare → single signals → amounts → categories → signals stack up → the test set is different. Bars show the **fraud rate** per group, and the grey line is the overall rate. This section is skipped when `train.csv` isn't available.

### 2.1 Fraud is rare: about 1 in 57 transactions
Always answering "not fraud" is 98.2% accurate, so accuracy is useless here. We judge the model by **PR-AUC** (how well frauds are ranked to the top), where random guessing scores only about 0.018.

In [4]:
def fraud_rate(by):
    """Rows, frauds, fraud rate (%) and lift (x the overall rate) per group of the training data."""
    g = train_clean.groupby(by, observed=True)["fraud"].agg(rows="size", frauds="sum")
    return g.assign(rate_pct=100 * g.frauds / g.rows, lift=g.frauds / g.rows / base)

def rate_bars(ax, table, title, horizontal=False, label=True):
    """Fraud rate per group as bars, with the overall rate as a grey reference line."""
    bars = (ax.barh if horizontal else ax.bar)([str(i) for i in table.index], table.rate_pct, 0.6, color=BLUE)
    (ax.axvline if horizontal else ax.axhline)(100 * base, color=MUTED, lw=1)
    if label:
        ax.bar_label(bars, fmt="%.1f%%", padding=2, fontsize=8)
    if horizontal:
        ax.invert_yaxis()
    ax.grid(axis="y" if horizontal else "x", visible=False)
    ax.set_title(title)

if HAS_TRAIN:
    t = train_clean
    base = t.fraud.mean()
    print(f"{t.fraud.sum()} frauds in {len(t):,} rows = {base:.2%}. 'Never fraud' would be {1 - base:.2%} accurate.")

353 frauds in 20,000 rows = 1.76%. 'Never fraud' would be 98.23% accurate.


### 2.2 Single signals: risk is flat, then it jumps
Each signal is flat over most of its range and then jumps: amounts above 500, **4+ transactions in an hour** (13–21% fraud), **11+ in a day**, a **new device** (7.5% vs 1.2%, the strongest single signal), night hours and brand-new accounts. Jumps like these suit **threshold flags and tree models** better than straight lines.

In [5]:
if HAS_TRAIN:
    panels = [
        (pd.cut(t.transaction_amount, [0, 50, 100, 500, 1000, np.inf], right=False, labels=["<50", "50-100", "100-500", "500-1k", "1k+"]), "Amount: jump above 500"),
        (pd.cut(t.transactions_last_1h, [0, 1, 2, 4, 6, np.inf], right=False, labels=["0", "1", "2-3", "4-5", "6+"]), "Transactions in last hour: jump at 4+"),
        (pd.cut(t.transactions_last_24h, [0, 6, 11, 16, np.inf], right=False, labels=["1-5", "6-10", "11-15", "16+"]), "Transactions in last 24h: jump at 11+"),
        (t.new_device.map({0: "known", 1: "new"}), "Device: new is 6x riskier"),
        (pd.cut(t.account_age, [0, 31, 91, 366, 1096, np.inf], right=False, labels=["<=30", "31-90", "91-365", "366-1095", "1096+"]), "Account age: youngest riskiest"),
        (t.transaction_hour.astype("Int64"), "Hour of day: riskier at night (23-04)"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
    for ax, (groups, title) in zip(axes.flat, panels):
        rate_bars(ax, fraud_rate(groups.rename(None)), title, label=not title.startswith("Hour"))
    axes[1, 2].set_xticks(axes[1, 2].get_xticks()[::3])
    for ax in axes[:, 0]:
        ax.set_ylabel("fraud rate (%)")
    fig.suptitle(f"Fraud rate by single signal (grey line: overall {base:.1%})", x=0.01, ha="left", fontsize=12, weight="bold")
    fig.tight_layout()
    plt.show()

### 2.3 Fraudsters spend more, but the ranges overlap
Fraud amounts are typically about 3× larger, yet many frauds are small and many large payments are legitimate. Amount is a signal, not a rule. Outliers are kept (frauds live in the tail), and the model also gets `log(amount)`.

In [6]:
if HAS_TRAIN:
    bins = np.logspace(np.log10(2), np.log10(10_000), 25)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    for label, color, name in [(0, BLUE, "legitimate"), (1, ORANGE, "fraud")]:
        x = t.loc[t.fraud == label, "transaction_amount"].dropna()
        w = np.full(len(x), 1 / len(x))
        ax.hist(x, bins, weights=w, histtype="stepfilled", color=color, alpha=0.1)
        ax.hist(x, bins, weights=w, histtype="step", color=color, lw=2, label=f"{name} (median {x.median():,.0f})")
    ax.set_xscale("log")
    ax.set(xlabel="transaction amount (log scale)", ylabel="share of rows", title="Amount distribution: fraud vs legitimate")
    ax.legend()
    plt.show()

### 2.4 Categories: luxury, cash transfers and electronics; Singapore is safest
The riskiest merchants are luxury, cash transfers and electronics, about 4–5× the safest (grocery, transport). Singapore has 70% of the rows and the lowest fraud rate of any country. Card-present is the safest channel.

In [7]:
if HAS_TRAIN:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, c in zip(axes, CATS):
        rate_bars(ax, fraud_rate(c).sort_values("rate_pct", ascending=False), c.replace("_", " "), horizontal=True)
        ax.set_xlabel("fraud rate (%)")
    fig.tight_layout()
    plt.show()

### 2.5 Signals stack up: combinations are where fraud concentrates
A new device or a burst alone is moderately risky. **Together** they are far riskier: a new device with 4+ transactions in the last hour is fraud roughly 1 time in 3. A model must learn these **interactions**, which is a key reason for a tree model (and for the `n_flags` feature in section 3).

In [8]:
if HAS_TRAIN:
    from matplotlib.colors import LinearSegmentedColormap

    burst = pd.cut(t.transactions_last_1h, [0, 1, 2, 4, 6, np.inf], right=False, labels=["0", "1", "2-3", "4-5", "6+"])
    device = t.new_device.map({0: "known device", 1: "new device"})
    cells = t.groupby([device, burst], observed=True)["fraud"].agg(["mean", "size"])
    rate, rows = 100 * cells["mean"].unstack(), cells["size"].unstack()
    fig, ax = plt.subplots(figsize=(8, 2.6))
    im = ax.imshow(rate.values, cmap=LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#0d366b"]), aspect="auto")
    for (i, j), v in np.ndenumerate(rate.values):
        ax.text(j, i, f"{v:.1f}%\nn={rows.values[i, j]:,}", ha="center", va="center", fontsize=8,
                color="white" if v > rate.values.max() / 2 else "#0b0b0b")
    ax.set_xticks(range(rate.shape[1]), rate.columns)
    ax.set_yticks(range(rate.shape[0]), rate.index)
    ax.grid(False)
    ax.set(xlabel="transactions in the last hour", title="Fraud rate: device x transactions in the last hour")
    fig.colorbar(im, ax=ax, label="fraud rate (%)")
    plt.show()

### 2.6 The test set is different (distribution shift)
The test set has about 3× more blanks, more new devices and bursts, and far more large amounts (mostly outside Singapore). The same signals are also **fraud less often** there: CV PR-AUC 0.238 vs public 0.196. Every correction we tried for this lost on the leaderboard, so the model is kept plain.

In [9]:
if HAS_TRAIN:
    checks = {"rows with a blank": lambda d: d[RAW_COLS].isna().any(axis=1).mean(),
              "new device": lambda d: d.new_device.mean(),
              "4+ transactions in 1h": lambda d: (d.transactions_last_1h >= 4).mean(),
              "amount > 1,000": lambda d: (d.transaction_amount > 1000).mean(),
              "Singapore": lambda d: (d.country == "SG").mean()}
    shares = pd.DataFrame({name: {k: 100 * f(d) for k, f in checks.items()} for name, d in [("train", train_clean), ("test", test_clean)]})
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 1.2]})
    y = np.arange(len(shares))
    for k, (name, color) in enumerate([("train", BLUE), ("test", ORANGE)]):
        bars = ax1.barh(y + (k - 0.5) * 0.4, shares[name], 0.36, color=color, label=name)
        ax1.bar_label(bars, fmt="%.1f%%", padding=2, fontsize=8)
    ax1.set_yticks(y, shares.index)
    ax1.invert_yaxis()
    ax1.grid(axis="y", visible=False)
    ax1.set(xlabel="% of rows", title="Share of rows: train vs test")
    ax1.legend()
    bins = np.logspace(np.log10(2), np.log10(10_000), 40)
    for name, d, color in [("train", train_clean, BLUE), ("test", test_clean, ORANGE)]:
        x = d.transaction_amount.dropna()
        ax2.hist(x, bins, weights=np.full(len(x), 1 / len(x)), histtype="step", color=color, lw=2, label=name)
    ax2.set_xscale("log")
    ax2.set(xlabel="transaction amount (log scale)", ylabel="share of rows", title="Amounts: test has a heavier large-amount tail")
    ax2.legend()
    fig.tight_layout()
    plt.show()

## 3. Feature engineering
24 features. Each uses only the row's own values (no labels, no other rows), so there is no leakage.

| Group | Features | Why |
|---|---|---|
| Raw (10) | the 10 input columns | The strongest signals live here |
| Transforms | `log_amt`, `log_spend`, `log_age`, `hour_sin`, `hour_cos` | Skewed amounts and ages; hour is cyclic (23:00 is next to 00:00) |
| Account-relative | `amt_to_avg_ticket`, `share_1h` | Fraud is what's unusual *for this account* |
| EDA flags | `night`, `burst`, `high24`, `young`, `risky_merchant`, `night_x_device`, `n_flags` | Encode the EDA thresholds and their combinations |

Categoricals go to CatBoost as text: its ordered target statistics encode them without leakage. Blank categories become `"Unknown"`.

In [10]:
def engineer(d):
    d = sanitise(d)
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

FEATURES = ["transaction_amount", "transaction_hour", "merchant_category", "country", "transaction_channel",
            "transactions_last_24h", "spend_last_24h", "account_age", "new_device", "transactions_last_1h",
            "log_amt", "log_spend", "log_age", "hour_sin", "hour_cos", "night", "burst", "high24", "young",
            "amt_to_avg_ticket", "share_1h", "risky_merchant", "night_x_device", "n_flags"]

CATEGORIES = {c: sorted(LEVELS[c] + ["Unknown"]) for c in CATS}   # training category lists

def test_features(frame):
    """Features for any test table: works with or without an id column, extra or missing columns."""
    Xt = engineer(frame.drop(columns=["id"], errors="ignore"))
    for c in CATS:                                # test categories use the training category lists
        Xt[c] = pd.Categorical(Xt[c].astype(str), categories=CATEGORIES[c])
    return Xt[FEATURES]

def as_catboost(d):                               # CatBoost takes categoricals as text
    return d.astype({c: str for c in CATS})

X_test = test_features(test)
print("Test features:", X_test.shape)

Test features: (12000, 24)


### 3.1 Checking the combination features
The fraud rate climbs steeply with `n_flags`, the number of risk flags fired: about 0.8% with none, about 16% with 3, and 40%+ with 4 or more. It also shows the limit: some frauds fire **no flag at all** ("silent fraud"), which is why the model keeps the raw, continuous inputs as well as the flags.

In [11]:
if HAS_TRAIN:
    n_flags = engineer(train).n_flags.clip(upper=5).astype(int)
    table = fraud_rate(n_flags.map(lambda k: "5+" if k == 5 else str(k)).rename(None))
    fig, ax = plt.subplots(figsize=(7, 3))
    rate_bars(ax, table, "Fraud rate by number of risk flags fired (n_flags)")
    ax.set_xticks(range(len(table)), [f"{i}\nn={r:,}" for i, r in zip(table.index, table.rows)])
    ax.set_ylabel("fraud rate (%)")
    plt.show()
    print(f"Silent fraud: {(n_flags[train.fraud == 1] == 0).mean():.0%} of frauds fire none of the flags.")

Silent fraud: 23% of frauds fire none of the flags.


## 4. Model: load the trained CatBoost
**Why CatBoost:** it learns the signal combinations where fraud concentrates, encodes categories without leakage, handles blanks natively, and scored best on the shifted public test set (comparison in section 6).

| Setting | Value | Why |
|---|---|---|
| depth | 4 | Combines up to 4 signals, but is too shallow to memorise 353 frauds |
| learning rate × iterations | 0.03 × 800 | Many small steps |
| L2 regularisation | 10 | Strong shrinkage against noise |
| class weights | none | Weighting cut CV PR-AUC by 0.016–0.04. Recall comes from the cut-off instead |
| seeds | 0, 1, 2, averaged | Less variance than one fit |

**Results (5-fold stratified CV, out-of-fold):** PR-AUC **0.238** (95% CI 0.194–0.292; random 0.018), ROC-AUC 0.775. At the top 1.65%: precision 0.315, recall 0.295, **F1 0.305**. **Public leaderboard PR-AUC: 0.196.**

The cell below loads `model.pkl` and checks that the constants above match the ones the models were trained with.

In [12]:
bundle = joblib.load("model.pkl")                  # saved by `python -m fraudlab train` (or notebooks/01)
models, share = bundle["models"], bundle["share"]
assert (bundle["features"], bundle["raw_cols"], bundle["non_negative"], bundle["cat_levels"], bundle["categories"]) == \
       (FEATURES, RAW_COLS, NON_NEGATIVE, CAT_LEVELS, CATEGORIES), "notebook constants differ from the trained model"
print(f"{len(models)} models, seeds {[m.get_params()['random_seed'] for m in models]}, flag share {share:.2%}")
print({k: models[0].get_params()[k] for k in ["iterations", "depth", "learning_rate", "l2_leaf_reg"]})

3 models, seeds [0, 1, 2], flag share 1.65%
{'iterations': 800, 'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 10}


### 4.1 Feature importance
The 10 most important features, averaged over the 3 models: amount, account age and `n_flags` lead.

In [13]:
importance = pd.Series(np.mean([m.get_feature_importance() for m in models], axis=0), index=FEATURES).sort_values().tail(10)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar_label(ax.barh(importance.index, importance.values, 0.6, color=BLUE), fmt="%.1f", padding=2, fontsize=8)
ax.grid(axis="y", visible=False)
ax.set(xlabel="importance (all 24 features sum to 100)", title="Top 10 features, mean of the 3 models")
plt.show()

## 5. Predictions
- **Score:** the average fraud probability over the 3 models.
- **Cut-off:** the top 1.65% of rows (the best-F1 share in CV) are mapped to ≥ 0.5 and the rest below. The map keeps the order, so PR-AUC is unchanged, while F1 and Recall at a 0.5 threshold match the CV-optimal point.
- **Output:** one row per test row, in the test file's order, with a single column `prediction` (the platform's format). If the test file has an `id` column (as locally), it is kept as well: `id, prediction`.

In [14]:
def to_submission_score(score, thr):
    """Monotone map to [0,1] that puts the flag cut-off at 0.5 (order, and so PR-AUC, unchanged)."""
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

score = np.mean([m.predict_proba(as_catboost(X_test))[:, 1] for m in models], axis=0)
score = np.where(np.isfinite(score), score, np.nanmedian(score) if np.isfinite(score).any() else 0.0)
preds = to_submission_score(score, np.quantile(score, 1 - share))
assert len(preds) == len(test) and np.isfinite(preds).all() and ((preds >= 0) & (preds <= 1)).all()

submission = pd.DataFrame({"prediction": preds})                # platform format: a single column, in the input row order
if "id" in test.columns:                                       # local test file with ids: keep them alongside
    submission.insert(0, "id", test["id"].values)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(submission)} predictions to {OUTPUT_PATH}; flagged (>= 0.5): {(preds >= 0.5).sum()}")
display(submission.head())

Wrote 12000 predictions to predictions.csv; flagged (>= 0.5): 198


,id,prediction
0,FR021005,0.009953
1,FR030078,0.007503
2,FR020717,0.020187
3,FR026657,0.021016
4,FR023775,0.008796


### 5.1 What the submitted scores look like
Most rows score near 0. Exactly the top 1.65% sit above the 0.5 line: these are the rows flagged as fraud for F1 and Recall.

In [15]:
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(preds, bins=50, color=BLUE, rwidth=0.9)
ax.set_yscale("log")
ax.axvline(0.5, color=MUTED, lw=1)
ax.text(0.52, ax.get_ylim()[1] / 3, f"flagged: {(preds >= 0.5).sum():,} rows ({(preds >= 0.5).mean():.2%})", fontsize=8)
ax.grid(axis="x", visible=False)
ax.set(xlabel="prediction", ylabel="rows (log scale)", title="Distribution of the submitted scores")
plt.show()

## 6. What else was tried, and why this model was kept
All alternatives were judged on the public leaderboard (12,000 test rows, ~212 frauds). The same model moves ±0.004 just from the choice of seeds, so only bigger differences mean anything.

| Alternative | Public PR-AUC | Lesson |
|---|---|---|
| **This model** (CatBoost, 3 seeds) | **0.19586** | – |
| TabPFN + LightGBM + logistic blend | 0.19354 | More models did not help |
| CatBoost 1 seed / 10 seeds | 0.1925 / 0.1921 | Seed count is noise, not a lever |
| LightGBM | 0.1901 | – |
| Shift corrections (flag-excess correction + within-dataset context features) | 0.1893 | Reweighting signals for the test mix didn't help |
| Unlabelled test rows added as "not fraud" | 0.1878 | The test's extra flagged rows are not simply legitimate |
| TabPFN | 0.1875 | – |
| CatBoost + logistic 50/50 | 0.1832 | Logistic regression gives every flag a fixed bonus, which misfires on the shifted test set |
| Monotone-constrained CatBoost | ~0.15 | Forcing "more signal = more risk" everywhere amplified the misfiring flags |

Rejected on cross-validation before any upload:
- class weighting (−0.016 to −0.04);
- target / WoE encoding, missing-value indicators, splines and explicit interactions (all within noise);
- depth 3 / 5 and stronger regularisation;
- relative "context" features (−0.03);
- kNN / neural-net ensembles (−0.06).

**Model assumptions:**
1. **Rows are independent.** There is no account id or timestamp, so sequences can't be modelled. Repeat-account collisions occur only at the chance level.
2. **The training labels are correct.**
3. **The feature–fraud link carries over to new data.** This is the assumption the train → test shift strains: the same flags are fraud less often in the test set, which is why every model scores ~0.045 lower there than in CV.
4. **Blanks carry no fraud signal.** Training rows with blanks held no fraud, and the test set has 3× more blanks. Blanks go to an `Unknown` level or CatBoost's native missing branch, and are never zero-filled.
5. **The organisers flag at 0.5 (unconfirmed).** The score map puts exactly the top 1.65% at ≥ 0.5 for F1 and Recall. PR-AUC doesn't depend on it.
